# Uber NCR Ride Analytics - Explanatory Data Analysis
**Prepared for:** Operations & Growth leadership, Uber Delhi NCR

## 1. Business & Visualization Context
- **Audience.** Operations and growth managers who decide on driver incentives, dispatch rules and fleet investment. They are data literate but busy, so every chart carries one message with a direct title and minimal clutter.
- **What we want them to know and do.** See *where booked demand leaks away before it becomes revenue*, and act on the three levers this analysis exposes: driver side cancellations, pickup wait times, and supply timing.
- **How data makes the point.** We rebuild the full booking funnel (dashboards usually show only completed rides), attach money to every leak, and slice demand by time to show when supply is mismatched.
- **The data.** 150,000 ride bookings across Delhi NCR for 2024, with 21 columns: booking outcome, vehicle type, pickup/drop zones, wait times (VTAT = vehicle arrival time), fares, distances, ratings, cancellation reasons and payment methods.

## 2. Data Exploration - Characteristics & Quality Issues
Profiling the raw file (next cells) reveals the issues that shape the cleaning plan:

1. **`"null"` strings instead of empty cells** - pandas treats them as missing on read, but it is worth verifying.
2. **IDs wrapped in extra quote characters** (`"CNR5884300"`) that must be stripped before customer level work.
3. **Missing values are structural, not random.** Fare, distance, rating and payment are empty for exactly the 48,000 bookings that never finished, and reason columns are filled only for cancelled rows. Dropping every row with a missing value would delete the most interesting part of the data, so we subset by outcome instead of imputing.
4. **Some columns are simulated flat.** Fare is uncorrelated with distance (scatter below), so per km pricing analysis would mislead. We focus on what genuinely varies: outcomes, timing, volume and mix.
5. **No duplicate booking IDs** and dates parse cleanly across the whole year, so the time axis is trustworthy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

blue = '#0b5394'
grey = '#c9ced6'
line_grey = '#b0b4ba'
row_blue = '#e9f0f8'
plt.rcParams['figure.facecolor'] = '#f9fafb'
plt.rcParams['axes.facecolor'] = '#f9fafb'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlelocation'] = 'left'

df = pd.read_csv('ncr_ride_bookings.csv')
print(df.shape)
df.head(3)

In [ ]:
print(df['Booking Status'].value_counts())
print()
print('Duplicated Booking IDs:', df['Booking ID'].duplicated().sum())
print()
print(df.isna().sum().sort_values(ascending=False).head(8))
done = df[df['Booking Status'] == 'Completed']
print()
print('Fare vs distance correlation:', round(done['Booking Value'].corr(done['Ride Distance']), 3))

In [ ]:
sample = done.sample(2000, random_state=42)
plt.figure(figsize=(8, 3.5))
plt.scatter(sample['Ride Distance'], sample['Booking Value'], s=10, color=blue, alpha=0.25)
plt.title('Fare does not depend on distance (r = 0.006)')
plt.xlabel('Ride distance (km)')
plt.ylabel('Booking value (INR)')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing
We strip the extra quotes from the ID columns, build a proper datetime with an hour feature, keep the reason columns as they are (their missing values carry meaning), and create a `completed` subset for all revenue work.

In [ ]:
df['Booking ID'] = df['Booking ID'].str.strip('"')
df['Customer ID'] = df['Customer ID'].str.strip('"')
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df['hour'] = df['datetime'].dt.hour
completed = df[df['Booking Status'] == 'Completed'].copy()
print('Completed rides:', len(completed))
print('Total revenue (INR millions):', round(completed['Booking Value'].sum() / 1e6, 1))

## 4. Insights
### 4.1 Only 62% of bookings ever become a ride
**Business importance:** every failed booking is demand the company already paid to attract (marketing, app sessions) that produced no revenue and hurt rider trust. Leadership normally looks at completed ride dashboards the funnel is where the money leaks.

In [ ]:
status = df['Booking Status'].value_counts().sort_values()
colors = []
for s in status.index:
    if s == 'Completed':
        colors.append(blue)
    else:
        colors.append(grey)
plt.figure(figsize=(8, 3.5))
plt.barh(status.index, status.values, color=colors)
for i, v in enumerate(status.values):
    plt.text(v + 1500, i, f'{v:,} ({v / len(df):.0%})', va='center')
plt.xlim(0, 112000)
plt.title('Only 62% of bookings become completed rides')
plt.xlabel('Bookings in 2024')
plt.tight_layout()
plt.show()

The ranked bars separate one blue success from four grey failures. **57,000 bookings (38%) never became rides.** At the average completed fare of INR 508, that is roughly **29M INR of unrealised revenue against the 47.3M actually earned**. The biggest growth opportunity in NCR is not new demand  it is demand we already had and lost.

### 4.2 Drivers cancel 2.6x more rides than customers
**Business importance:** rider side and driver side cancellations need opposite fixes (app UX versus incentives and policy). Knowing which side owns the problem decides where the budget goes.

In [ ]:
driver = df['Driver Cancellation Reason'].value_counts()
customer = df['Reason for cancelling by Customer'].value_counts()
reasons = pd.concat([driver, customer]).sort_values()
colors = []
for r in reasons.index:
    if r in driver.index:
        colors.append(blue)
    else:
        colors.append(grey)
plt.figure(figsize=(8, 4))
plt.barh(reasons.index, reasons.values, color=colors)
plt.title('Every driver reason (blue) outranks every customer reason')
plt.xlabel('Cancelled bookings')
plt.tight_layout()
plt.show()
print('Driver cancellations:', driver.sum(), '| Customer cancellations:', customer.sum())

Both reason columns are merged into one ranked chart, colour coded by who cancelled. Drivers killed **27,000 bookings 18% of all demand and 2.6x the customer total** and their four reasons are almost evenly spread, which points to a broad incentive problem rather than one fixable defect. Worse, two of the top *customer* reasons ("driver is not moving towards pickup", "driver asked to cancel") are also driver induced, so the true driver share is even higher. Cancellation is a **supply discipline problem**, not a rider problem.

### 4.3 Waiting kills bookings: the four minute cliff
**Business importance:** pickup wait (VTAT) is one of the few levers dispatch controls directly, through matching radius and honest ETAs. If long waits precede cancellations, tightening dispatch pays for itself.

In [ ]:
wait = df.groupby('Booking Status')['Avg VTAT'].mean().dropna().sort_values()
colors = []
for s in wait.index:
    if s == 'Cancelled by Customer':
        colors.append(blue)
    else:
        colors.append(grey)
plt.figure(figsize=(7.5, 3))
plt.barh(wait.index, wait.values, color=colors)
for i, v in enumerate(wait.values):
    plt.text(v + 0.15, i, f'{v:.1f} min', va='center')
plt.axvline(wait['Completed'], color='#555555', linestyle='--', linewidth=1)
plt.xlim(0, 14.5)
plt.title('Riders who cancelled waited 47% longer for their car')
plt.xlabel('Average vehicle arrival time (minutes)')
plt.tight_layout()
plt.show()

Average arrival time per outcome, with the completed ride benchmark (8.5 min) as a dashed line: bookings the **customer cancelled averaged 12.5 minutes a four minute, 47% penalty**, consistent with a patience threshold near the ten minute mark. The story is actionable: customer cancellations (10,500 rides, about 5.3M INR) are not fickleness, they are a **dispatch quality symptom**. Capping promised ETAs and re matching stalled pickups attacks them directly.

### 4.4 Two rush hours and a dead zone: demand swings 9x
**Business importance:** driver supply is recruited in shifts. If incentives do not mirror the real demand curve, the company pays for idle drivers at 4 AM and loses surge-priced rides at 6 PM.

In [ ]:
hourly = df.groupby('hour').size()
plt.figure(figsize=(9, 3.5))
plt.plot(hourly.index, hourly.values, color=line_grey, linewidth=2, marker='o', markersize=4)
plt.scatter([18], [hourly[18]], color=blue, s=90, zorder=3)
plt.annotate(f'Evening peak: {hourly[18]:,}', (18, hourly[18]), xytext=(15.3, hourly[18] + 700), color=blue, fontweight='bold')
plt.annotate(f'Morning peak: {hourly[10]:,}', (10, hourly[10]), xytext=(8.2, hourly[10] + 800), color='#555555')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.xticks(range(0, 24, 2))
plt.ylim(0, 14500)
plt.title('Demand peaks twice a day and collapses 9x overnight')
plt.xlabel('Hour of day')
plt.ylabel('Bookings')
plt.tight_layout()
plt.show()

The line chart exposes a **double peak day**: a 10:00 business crest of 9,577 bookings and a taller 18:00 peak of 12,397 (highlighted in blue), against a trough of just 1,321 at 4:00 a **9.4x swing**. Because the cancellation *rate* is flat across hours, the *absolute* number of failed bookings peaks exactly at rush hour, the worst possible moment to disappoint riders. Supply incentives should concentrate on the 17:00-19:00 window, where each recovered driver hour covers the most bookings.

### 4.5 The workhorse fleet: autos out earn every premium product
**Business importance:** fleet expansion budgets often chase premium categories. The real revenue mix should decide where onboarding money goes.

In [ ]:
veh = completed.groupby('Vehicle Type')['Booking Value'].agg(['count', 'sum', 'mean'])
veh = veh.sort_values('sum', ascending=False)
rows = []
for name, r in veh.iterrows():
    rows.append([name, f"{int(r['count']):,}", f"{r['sum'] / 1e6:.1f}M", f"{r['mean']:.0f}"])
fig, ax = plt.subplots(figsize=(8, 2.9))
ax.axis('off')
cols = ['Vehicle Type', 'Completed Rides', 'Revenue (INR)', 'Avg Fare (INR)']
tbl = ax.table(cellText=rows, colLabels=cols, cellLoc='center', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.7)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor('#e0e0e0')
    if r == 0:
        cell.set_facecolor(blue)
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor(row_blue)
ax.set_title('Completed ride revenue by vehicle type', fontweight='bold', loc='left')
plt.tight_layout()
plt.show()

The summary table ranks vehicle classes by completed ride revenue: the humble **Auto is the single largest earner (11.7M INR)**, and the three everyday classes Auto, Go Mini, Go Sedan deliver **63% of total revenue**, while the premium tier (Premier Sedan plus Uber XL) contributes only 15%. The average fare column shows why: fares are flat across classes, so revenue here is purely a *volume* game. Driver onboarding and retention spend should defend the workhorse categories first and treat premium as a margin experiment, not the growth engine.

### 4.6 Cash is a minority but still one rupee in four
**Business importance:** cash rides carry reconciliation cost, driver safety risk and no automatic commission collection. The digital adoption gap is a quantifiable cost line.

In [ ]:
payment = completed['Payment Method'].value_counts()
share = payment / payment.sum() * 100
colors = []
for name in share.index:
    if name == 'Cash':
        colors.append(blue)
    else:
        colors.append(grey)
plt.figure(figsize=(8, 3.5))
plt.bar(share.index, share.values, color=colors, width=0.6)
for i, v in enumerate(share.values):
    plt.text(i, v + 1, f'{v:.0f}%', ha='center')
plt.ylim(0, 52)
plt.ylabel('Share of completed rides (%)')
plt.title('UPI dominates, yet 25% of rides are still settled in cash')
plt.tight_layout()
plt.show()

The bars rank payment methods by share of completed rides: **UPI already carries 45%** and all digital rails together carry 75%, but **cash (blue) persists at 25% about 5.8M INR of fares handled physically**. Average fares are identical across methods, so cash users are not a distinct spending segment they are simply unconverted. A small wallet credit offer on a cash rider's next trip is the cheapest experiment this analysis suggests, with a directly measurable conversion metric.

## 5. Discussion & Conclusion
**Strengths of the pipeline.** It analyses the *whole funnel* rather than only completed rides every chart carries one pre attentive message (single accent colour, ranked bars, direct titles) and the preprocessing respects structural missingness instead of destroying it.

**Limitations.** The dataset appears synthetic: fares are independent of distance, ratings and distances are uniform across vehicle types, and seasonal variation is absent, so pricing and quality questions cannot be answered here. It covers one year and one region, with no driver IDs or geo coordinates for spatial analysis.

**Business implications & recommendations.**
1. **Attack driver cancellations first** (18% of all bookings): completion linked bonuses and penalties for cancelling after acceptance up to 13.7M INR of bookings are recoverable.
2. **Cap and honour ETAs.** Customer cancellations cluster behind ten minute plus waits tighten the dispatch radius at peak and auto rematch stalled pickups.
3. **Shape supply to the double peak.** Shift incentives into 17:00-19:00, where absolute failures are largest, and use a small scheduled pool for the pre dawn trough.
4. **Defend the workhorse fleet.** Auto, Go Mini and Go Sedan earn 63% of revenue protect their driver retention before premium expansion.
5. **Convert cash riders** with first use wallet credits a quarter of settled fares still move outside digital rails.

*Recovering even half of the failed booking value (about 14.5M INR) would grow realised revenue by roughly 30% with zero new demand.*